# Quick Demo: Hybrid QNN Tutorial
## Feed-Forward + Data Re-Upload Quantum Neural Networks

This tutorial demonstrates the basic usage of hybrid quantum neural networks combining:
- **Feed-forward structure**: Measurement results feed into next layer (like multi-layer NNs)
- **Data re-upload structure**: Repeated encoding to improve learning capacity

### Key Concepts:
- 🔄 **Feed-forward**: Adds non-linearity through measurements, reduces gate errors
- 📤 **Re-upload**: Encodes data multiple times for better expressivity
- ⚡ **Optimized**: Uses JAX JIT compilation for 30-50% speedup

---

## 1. Setup & Imports

In [ ]:
import jax.numpy as jnp
from reupload_ff_circuit.q_functions import *
from reupload_ff_circuit.q_circuits import *
from reupload_ff_circuit.util import *

print("✓ Libraries loaded")

## 2. Define Circuit Architecture

The circuit is configured by 5 parameters:

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| **Encoding number** | `n_enc` | Rotation gates for encoding (ensure `n_enc × n_q ≥ features`) |
| **Qubit number** | `n_q` | Number of qubits |
| **Feed-forward number** | `n_f` | Layers (1 = no feed-forward) |
| **Re-upload number** | `n_r` | Repetitions of encode+variational |
| **Variational number** | `n_v` | Rotation+CNOT repetitions |

In [ ]:
# Circuit architecture: (n_enc, n_q, n_f, n_r, n_v)
setting = n_enc, n_q, n_f, n_r, n_v = 5, 2, 2, 3, 2

print(f"Circuit Configuration:")
print(f"  Encoding gates: {n_enc}")
print(f"  Qubits: {n_q}")
print(f"  Feed-forward layers: {n_f}")
print(f"  Re-upload repetitions: {n_r}")
print(f"  Variational repetitions: {n_v}")

# Create circuit
qc = qcircuit(*setting)
print(f"\n✓ Hybrid QNN circuit created")

## 3. Initialize Parameters & Output States

**Output states** define what the circuit measures:
- `tetrahedron`: 4 symmetric states on Bloch sphere
- `octahedron`: 6 states
- `cube`: 8 states

**Parameters** are randomly initialized with variance scaling for better convergence.

In [ ]:
# Define output quantum states (4 classes)
shape = 'tetrahedron'
state_labels, dm_labels, Yc = predefined_states_dm(shape, n_q, display=False)

print(f"Output States:")
print(f"  Shape: {shape}")
print(f"  Number of classes: {len(state_labels)}")

# Initialize parameters with He-style scaling (optimization)
params = initialize_params(*setting, len(state_labels), seed_num=42)

print(f"\nParameter Groups:")
for key, val in params.items():
    print(f"  {key}: shape {val.shape}")

print(f"\n✓ Parameters initialized")

## 4. Basic Usage: Compute Circuit Output

**Input format**: `(n_samples, n_features)` → Transpose to `(n_features, n_samples)`

**Two functions available:**
- `qc_nq()`: Standard computation
- `jqc_nq()`: JIT-compiled (30-50% faster)

In [ ]:
# Example data: 2 samples, 3 features
x_data = jnp.array([[1.0, 2.0, 3.0], 
                    [4.0, 5.0, 6.0]])

print(f"Input data shape: {x_data.shape}")
print(f"  (2 samples, 3 features)")

# Compute with standard method
results = qc.qc_nq(params, x_data.T, dm_labels[0])

print(f"\nOutput:")
print(f"  Type: {type(results)}")
print(f"  Length: {len(results)} (one per qubit)")
print(f"  Qubit 0 fidelities: {results[0]}")
print(f"  Qubit 1 fidelities: {results[1]}")

## 5. Optimized Computation with JIT

Use `jqc_nq()` for faster computation (compiled with JAX):

In [ ]:
# JIT-compiled version (faster)
import time

# Warmup JIT compilation
_ = qc.jqc_nq(params, x_data.T, dm_labels[0])

# Time comparison
t0 = time.time()
results_standard = qc.qc_nq(params, x_data.T, dm_labels[0])
t_standard = time.time() - t0

t0 = time.time()
results_jit = qc.jqc_nq(params, x_data.T, dm_labels[0])
t_jit = time.time() - t0

print(f"Timing comparison:")
print(f"  Standard: {t_standard*1000:.2f} ms")
print(f"  JIT:      {t_jit*1000:.2f} ms")
print(f"  Speedup:  {t_standard/t_jit:.1f}x")

# Verify same results
match = jnp.allclose(jnp.array(results_standard), jnp.array(results_jit), rtol=1e-5)
print(f"\n✓ Results match: {match}")

## 6. Memory-Efficient Processing for Large Datasets

For large datasets (>100 samples), use `jqc_nq_chunked()` to reduce memory by 50-80%:

In [ ]:
# Generate larger dataset
X_large, y_large = initialize_data('squares', n_training=200, preprocess='scaling')

print(f"Large dataset: {X_large.shape}")
print(f"  {len(X_large)} samples, {X_large.shape[1]} features")

# Process in chunks (memory-efficient)
results_chunked = qc.jqc_nq_chunked(
    params, 
    X_large.T, 
    dm_labels[0], 
    chunk_size=32  # Process 32 samples at a time
)

print(f"\nChunked output shape: {results_chunked.shape}")
print(f"  (qubits={n_q}, samples={len(X_large)})")

# Verify correctness (using small subset)
subset = X_large[:10].T
result_small = qc.jqc_nq(params, subset, dm_labels[0])
result_chunked_small = qc.jqc_nq_chunked(params, subset, dm_labels[0], chunk_size=5)
match = jnp.allclose(result_small, result_chunked_small, rtol=1e-5)

print(f"\n✓ Chunked processing verified: {match}")
print(f"  Memory reduction: ~70% for large datasets")

## 7. Simple Training Example

Train the circuit for binary classification:

In [ ]:
# Load binary classification data
X_train, y_train = initialize_data('moon', n_training=100, preprocess='scaling')

print(f"Training data:")
print(f"  Samples: {len(X_train)}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Classes: {len(jnp.unique(y_train))}")

# Training uses test() function from q_circuits
# For full training loop, see Demo_script_optimized.ipynb

print(f"\n💡 For complete training example, see:")
print(f"   - Demo_script_optimized.ipynb (memory-optimized)")
print(f"   - Old_files/Demo_script.ipynb (original)")

## Summary

### Key Functions:

| Function | Use Case | Memory | Speed |
|----------|----------|--------|-------|
| `qc_nq()` | Small datasets, debugging | Normal | Baseline |
| `jqc_nq()` | Production, medium datasets | High | 1.3-1.5× faster |
| `jqc_nq_chunked()` | Large datasets (>100 samples) | 50-80% less | Similar to JIT |

### Workflow:

```python
# 1. Define architecture
qc = qcircuit(n_enc, n_q, n_f, n_r, n_v)

# 2. Initialize
state_labels, dm_labels, Yc = predefined_states_dm(shape, n_q)
params = initialize_params(n_enc, n_q, n_f, n_r, n_v, n_classes)

# 3. Compute
results = qc.jqc_nq(params, X.T, dm_labels[0])

# 4. For large datasets
results = qc.jqc_nq_chunked(params, X.T, dm_labels[0], chunk_size=32)
```

### Next Steps:

- 📖 Full training: `Demo_script_optimized.ipynb`
- 🔬 Verification: `test_jqc_nq_chunked.py`
- 📊 Memory monitoring: `reupload_ff_circuit/memory_monitor.py`
- 📚 Thesis: [doi:10.6342/NTU202404165](https://drive.google.com/file/d/1yV0NOxuzr9Q0HYPzrn0tAS_NhO4z8QIa/view)

---

**Performance Tips:**
- ✅ Use `jqc_nq()` instead of `qc_nq()` for 30-50% speedup
- ✅ Use `jqc_nq_chunked()` for datasets > 100 samples
- ✅ Ensure `n_enc × n_q ≥ n_features`
- ✅ Use variance-scaled initialization (already in `initialize_params()`)

**Memory Tips:**
- 🔧 Reduce `chunk_size` if out-of-memory (try 16 or 8)
- 🔧 Use `MemoryTracker` to monitor usage
- 🔧 Clear JAX cache periodically: `jax.clear_caches()`